# Social Graph Mining and Emotional Attribution using XAI
## Improved Aigents Pipeline with CORRECTED Exponential Strength Formula

**Research Goal**: Extract emotion-attributed social graphs from DialogRE conversations

**Key Improvements**:
1. ✅ **Corrected strength calculation**: Uses Pygents exponential formula `strength = 1 - exp(-1.5 × raw_score)`
2. ✅ **Multi-word boost**: Accounts for texts with multiple emotion words
3. ✅ **Confidence-based neutral handling**: Uses score margins and strength thresholds
4. ✅ **Enhanced evaluation**: F1, precision, recall with proper handling of 3-class to 2-class conversion
5. ✅ **Complete pipeline**: Text → Relations → Emotional Graph (JSON + PNG)

**Score Ranges (After Exponential Transform)**:
- Single emotion word: 0.53-0.78
- Multiple emotion words: 0.85-0.95
- Neutral (weak signals): 0.0-0.4


## 1. Setup and Imports

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, 
    f1_score, 
    precision_score, 
    recall_score,
    confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

# Setup paths (adjust as needed)
# cwd = os.getcwd()
# project_path = cwd[:cwd.find('pygents')+7] if 'pygents' in cwd else cwd
# if project_path not in sys.path:
#     sys.path.append(project_path)
# os.chdir(project_path)

# Mock Pygents API for demonstration (replace with actual import)
class MockPygentsSentiment:
    def get_sentiment(self, text):
        # Returns (label, pos_score, neg_score)
        # Simulating actual Pygents behavior
        pos_words = ['good', 'love', 'great', 'adores', 'likes', 'wonderful', 'excellent', 'amazing']
        neg_words = ['bad', 'hate', 'terrible', 'hates', 'awful', 'worst', 'dull']
        
        text_lower = text.lower()
        pos_count = sum(1 for w in pos_words if w in text_lower)
        neg_count = sum(1 for w in neg_words if w in text_lower)
        
        pos_score = min(pos_count * 0.3, 1.0)
        neg_score = -min(neg_count * 0.3, 1.0)
        
        label = 'positive' if pos_score > abs(neg_score) else 'negative'
        return (label, pos_score, neg_score)

class MockTextMetrics:
    def get_sentiment_words(self, text):
        pos_words = ['good', 'love', 'great', 'adores', 'likes', 'wonderful']
        neg_words = ['bad', 'hate', 'terrible', 'hates', 'awful', 'worst']
        
        text_lower = text.lower()
        pos_count = sum(1 for w in pos_words if w in text_lower)
        neg_count = sum(1 for w in neg_words if w in text_lower)
        
        result = {}
        if pos_count > 0:
            result['positive'] = min(pos_count * 0.3, 1.0)
        if neg_count > 0:
            result['negative'] = min(neg_count * 0.3, 1.0)
        return result

# Initialize
# Uncomment when using actual Pygents:
# from pygents.aigents_api import PygentsSentiment, TextMetrics
# p = PygentsSentiment('./data/dict/en/positive.txt',
#                      './data/dict/en/negative.txt', debug=False)
# sentiment_metrics = TextMetrics({
#     'positive': './data/dict/en/positive.txt',
#     'negative': './data/dict/en/negative.txt'
# }, debug=False)

# Using mock for demonstration
p = MockPygentsSentiment()
sentiment_metrics = MockTextMetrics()

print("✓ Sentiment analyzer initialized")

## 2. Load and Prepare Data

In [ ]:
# Load the annotated CSV
csv_path = '/mnt/user-data/uploads/annotation_samples_150_-_pygents_samples_200.csv'
df_gold = pd.read_csv(csv_path)

# Rename column for consistency
if 'manual_label' in df_gold.columns:
    df_gold.rename(columns={'manual_label': 'gold_label'}, inplace=True)

# Map subject_name and object_name if they exist
if 'subject_name' in df_gold.columns and 'object_name' in df_gold.columns:
    df_gold['subject'] = df_gold['subject_name']
    df_gold['object'] = df_gold['object_name']

print(f"✓ Loaded {len(df_gold)} annotated samples")
print(f"\nLabel distribution:")
print(df_gold['gold_label'].value_counts())
print(f"\nSample:")
print(df_gold[['subject', 'object', 'sentence', 'gold_label']].head(3))

## 3. Neutral Handling Strategy

### Problem
Aigents is a binary classifier (positive/negative) but our annotations include neutral labels.

### Solution: Confidence-Based Neutral Detection

**Strategy:**
1. **Score Margin Threshold**: If |pos_score - |neg_score|| < threshold → likely neutral
2. **Absolute Strength Threshold**: If both scores are very low → likely neutral
3. **Confidence Score**: Combine both signals for final decision

**Implementation:**
```python
margin = |pos_score - |neg_score||
max_score = max(pos_score, |neg_score|)

if margin < MARGIN_THRESHOLD and max_score < STRENGTH_THRESHOLD:
    → Treat as neutral (exclude from binary evaluation)
else:
    → Use Aigents prediction (positive or negative)
```

**Tunable Parameters:**
- `MARGIN_THRESHOLD`: How similar must pos/neg scores be? (0.1-0.3)
- `STRENGTH_THRESHOLD`: How weak must both signals be? (0.2-0.4)


In [ ]:
def calculate_confidence_and_strength(pos_score, neg_score):
    """
    Calculate confidence scores and strength for neutral detection.
    Uses Pygents' standard exponential formula for strength calculation.
    
    Args:
        pos_score: Positive score from Pygents (raw score, typically 0-2)
        neg_score: Negative score from Pygents (usually negative, -2 to 0)
    
    Returns:
        dict with: label, strength, confidence, margin, max_score, is_likely_neutral
    """
    # Normalize negative score to positive range
    neg_score_abs = abs(neg_score)
    
    # Determine label and raw strength based on dominant score
    if pos_score > neg_score_abs:
        label = 'positive'
        raw_strength = pos_score
    else:
        label = 'negative'
        raw_strength = neg_score_abs
    
    # ========== PYGENTS STANDARD FORMULA ==========
    # Apply exponential strength transformation
    # Formula: strength = 1 - exp(-1.5 × raw_strength)
    # Maps: 0.3→0.36, 0.5→0.53, 1.0→0.78, 1.5→0.89
    strength = 1 - np.exp(-1.5 * raw_strength)
    
    # Multi-word boost (optional - for texts with multiple emotion words)
    total_emotion = pos_score + neg_score_abs
    if total_emotion > 1.0:
        boost = min(0.15, (total_emotion - 1.0) * 0.1)
        strength = min(1.0, strength + boost)
    # ==============================================
    
    # Calculate margin (difference between scores)
    margin = abs(pos_score - neg_score_abs)
    
    # Maximum score magnitude
    max_score = max(pos_score, neg_score_abs)
    
    # Calculate confidence (0-1)
    # High confidence = large margin relative to total
    if (pos_score + neg_score_abs) > 0:
        confidence = margin / (pos_score + neg_score_abs)
    else:
        confidence = 0.0
    
    # Neutral detection thresholds (tunable)
    MARGIN_THRESHOLD = 0.2  # Tunable: 0.1-0.3
    STRENGTH_THRESHOLD = 0.3  # Tunable: 0.2-0.4 (applied to raw_strength)
    
    is_likely_neutral = (margin < MARGIN_THRESHOLD) and (raw_strength < STRENGTH_THRESHOLD)
    
    return {
        'label': label,
        'strength': strength,  # Now using exponential formula!
        'raw_strength': raw_strength,  # Keep raw for debugging
        'confidence': confidence,
        'margin': margin,
        'max_score': max_score,
        'is_likely_neutral': is_likely_neutral
    }

# Test the function
test_cases = [
    (0.5, -0.0, "Single positive word (likes)"),
    (1.0, -0.0, "Strong positive word (hates)"),
    (1.5, -0.0, "Multiple positive words (loves and adores)"),
    (0.15, -0.12, "Likely neutral - weak signals"),
]

print("Testing neutral detection strategy with EXPONENTIAL STRENGTH:\n")
for pos, neg, desc in test_cases:
    result = calculate_confidence_and_strength(pos, neg)
    print(f"{desc}:")
    print(f"  Raw scores: pos={pos:.2f}, neg={neg:.2f}")
    print(f"  Result: {result['label']}, raw_strength={result['raw_strength']:.3f}, strength={result['strength']:.3f}")
    print(f"  Confidence: {result['confidence']:.3f}, Margin: {result['margin']:.3f}")
    print(f"  Is neutral? {result['is_likely_neutral']}")
    print()



## 4. Predict Sentiments with Pygents

In [ ]:
def predict_sentiment(text):
    """
    Predict sentiment using Pygents API with neutral detection.
    """
    # Get Pygents prediction
    sentiment = p.get_sentiment(text)
    label_raw = sentiment[0]
    pos_score = sentiment[1]
    neg_score = sentiment[2]
    
    # Get word-level metrics
    word_metrics = sentiment_metrics.get_sentiment_words(text)
    
    # Calculate confidence and detect neutrals
    analysis = calculate_confidence_and_strength(pos_score, neg_score)
    
    return {
        'pred_label': analysis['label'],
        'pos_score_raw': pos_score,
        'neg_score_raw': neg_score,
        'strength': analysis['strength'],
        'confidence': analysis['confidence'],
        'margin': analysis['margin'],
        'max_score': analysis['max_score'],
        'is_likely_neutral': analysis['is_likely_neutral'],
        'pos_words': word_metrics.get('positive', 0),
        'neg_words': word_metrics.get('negative', 0)
    }

# Run predictions
print("Running Pygents predictions...\n")
predictions = []
for idx, row in df_gold.iterrows():
    pred = predict_sentiment(row['sentence'])
    predictions.append(pred)

# Add predictions to dataframe
pred_df = pd.DataFrame(predictions)
df_results = pd.concat([df_gold.reset_index(drop=True), pred_df], axis=1)

print(f"✓ Completed {len(df_results)} predictions")
print(f"\nPredicted distribution:")
print(df_results['pred_label'].value_counts())
print(f"\nDetected likely neutrals: {df_results['is_likely_neutral'].sum()}")

## 5. Evaluation Strategy

### Challenge
Our gold labels have 3 classes (positive, negative, neutral), but Pygents predicts 2 classes.

### Solution: Multiple Evaluation Modes

**Mode 1: Exclude Neutrals**
- Only evaluate on gold positive/negative samples
- Most fair for binary classifier
- Shows true performance on polarized samples

**Mode 2: Map Neutrals to Dominant Class**
- Map gold neutral → negative (more samples) OR
- Map to predicted class (treat as uncertain)
- Tests robustness to neutral misclassification

**Mode 3: Exclude Detected Neutrals**
- Remove samples where Pygents detected likely neutral
- Focuses on high-confidence predictions
- Shows best-case performance


In [ ]:
def evaluate_predictions(df, mode='exclude_gold_neutral'):
    """
    Evaluate predictions with different handling of neutral labels.
    
    Args:
        df: DataFrame with gold_label and pred_label columns
        mode: 'exclude_gold_neutral', 'map_neutral_to_neg', 'exclude_detected_neutral'
    
    Returns:
        dict with metrics
    """
    df_eval = df.copy()
    
    if mode == 'exclude_gold_neutral':
        # Only evaluate on gold positive/negative samples
        df_eval = df_eval[df_eval['gold_label'].isin(['positive', 'negative'])]
        print(f"Mode: Exclude gold neutral samples")
        print(f"Evaluating on {len(df_eval)}/{len(df)} samples\n")
    
    elif mode == 'map_neutral_to_neg':
        # Map gold neutral to negative (majority class)
        df_eval['gold_label'] = df_eval['gold_label'].replace('neutral', 'negative')
        print(f"Mode: Map gold neutral → negative")
        print(f"Evaluating on all {len(df_eval)} samples\n")
    
    elif mode == 'exclude_detected_neutral':
        # Exclude samples we detected as likely neutral
        df_eval = df_eval[df_eval['is_likely_neutral'] == False]
        # Also exclude gold neutrals
        df_eval = df_eval[df_eval['gold_label'].isin(['positive', 'negative'])]
        print(f"Mode: Exclude detected neutrals + gold neutrals")
        print(f"Evaluating on {len(df_eval)}/{len(df)} samples (high confidence only)\n")
    
    if len(df_eval) == 0:
        print("⚠️  No samples to evaluate!")
        return {}
    
    y_true = df_eval['gold_label'].values
    y_pred = df_eval['pred_label'].values
    
    # Calculate metrics
    accuracy = (y_true == y_pred).mean()
    
    # F1, Precision, Recall
    f1_pos = f1_score(y_true, y_pred, pos_label='positive', zero_division=0)
    f1_neg = f1_score(y_true, y_pred, pos_label='negative', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    precision_pos = precision_score(y_true, y_pred, pos_label='positive', zero_division=0)
    precision_neg = precision_score(y_true, y_pred, pos_label='negative', zero_division=0)
    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    
    recall_pos = recall_score(y_true, y_pred, pos_label='positive', zero_division=0)
    recall_neg = recall_score(y_true, y_pred, pos_label='negative', zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=['positive', 'negative'])
    
    results = {
        'mode': mode,
        'n_samples': len(df_eval),
        'accuracy': accuracy,
        'f1_positive': f1_pos,
        'f1_negative': f1_neg,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_positive': precision_pos,
        'precision_negative': precision_neg,
        'precision_macro': precision_macro,
        'recall_positive': recall_pos,
        'recall_negative': recall_neg,
        'recall_macro': recall_macro,
        'confusion_matrix': cm,
        'df_eval': df_eval
    }
    
    return results

def print_evaluation(results):
    """Pretty print evaluation results."""
    print("=" * 70)
    print(f"EVALUATION RESULTS - {results['mode']}")
    print("=" * 70)
    print(f"\nSamples evaluated: {results['n_samples']}")
    print(f"\nAccuracy: {results['accuracy']:.3f}")
    print(f"\n--- Per-Class Metrics ---")
    print(f"Positive: F1={results['f1_positive']:.3f}, P={results['precision_positive']:.3f}, R={results['recall_positive']:.3f}")
    print(f"Negative: F1={results['f1_negative']:.3f}, P={results['precision_negative']:.3f}, R={results['recall_negative']:.3f}")
    print(f"\n--- Aggregate Metrics ---")
    print(f"F1 Macro: {results['f1_macro']:.3f}")
    print(f"F1 Weighted: {results['f1_weighted']:.3f}")
    print(f"Precision Macro: {results['precision_macro']:.3f}")
    print(f"Recall Macro: {results['recall_macro']:.3f}")
    print(f"\n--- Confusion Matrix ---")
    print("              Pred Positive  Pred Negative")
    print(f"Gold Positive      {results['confusion_matrix'][0][0]:6d}         {results['confusion_matrix'][0][1]:6d}")
    print(f"Gold Negative      {results['confusion_matrix'][1][0]:6d}         {results['confusion_matrix'][1][1]:6d}")
    print()

# Run all evaluation modes
print("\n" + "#" * 70)
print("#  COMPREHENSIVE EVALUATION")
print("#" * 70 + "\n")

eval_results = {}

# Mode 1: Exclude gold neutrals (most fair)
results1 = evaluate_predictions(df_results, mode='exclude_gold_neutral')
print_evaluation(results1)
eval_results['exclude_gold_neutral'] = results1

# Mode 2: Map neutrals to negative
results2 = evaluate_predictions(df_results, mode='map_neutral_to_neg')
print_evaluation(results2)
eval_results['map_neutral_to_neg'] = results2

# Mode 3: Exclude detected neutrals (best case)
results3 = evaluate_predictions(df_results, mode='exclude_detected_neutral')
print_evaluation(results3)
eval_results['exclude_detected_neutral'] = results3

## 6. Comparison Summary

In [ ]:
# Create comparison table
comparison_data = []
for mode, results in eval_results.items():
    comparison_data.append({
        'Mode': mode,
        'N_Samples': results['n_samples'],
        'Accuracy': f"{results['accuracy']:.3f}",
        'F1_Macro': f"{results['f1_macro']:.3f}",
        'F1_Pos': f"{results['f1_positive']:.3f}",
        'F1_Neg': f"{results['f1_negative']:.3f}",
        'Precision': f"{results['precision_macro']:.3f}",
        'Recall': f"{results['recall_macro']:.3f}"
    })

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "=" * 70)
print("COMPARISON ACROSS EVALUATION MODES")
print("=" * 70)
print(df_comparison.to_string(index=False))
print("\n⭐ Recommended: 'exclude_gold_neutral' - most fair for binary classifier")
print("⭐ Best case: 'exclude_detected_neutral' - high confidence predictions only")

## 7. Error Analysis

In [ ]:
def analyze_errors(df_eval, top_n=5):
    """
    Detailed error analysis.
    """
    errors = df_eval[df_eval['gold_label'] != df_eval['pred_label']].copy()
    correct = df_eval[df_eval['gold_label'] == df_eval['pred_label']].copy()
    
    print("=" * 70)
    print("ERROR ANALYSIS")
    print("=" * 70)
    print(f"\nTotal errors: {len(errors)}/{len(df_eval)} ({len(errors)/len(df_eval)*100:.1f}%)")
    
    if len(errors) == 0:
        print("\n🎉 Perfect predictions!")
        return
    
    # Error breakdown
    print("\n--- Error Breakdown ---")
    fp = len(errors[(errors['gold_label']=='negative') & (errors['pred_label']=='positive')])
    fn = len(errors[(errors['gold_label']=='positive') & (errors['pred_label']=='negative')])
    print(f"False Positives (pred pos, gold neg): {fp}")
    print(f"False Negatives (pred neg, gold pos): {fn}")
    
    # Confidence analysis
    print("\n--- Confidence Analysis ---")
    print(f"Avg confidence of errors: {errors['confidence'].mean():.3f}")
    print(f"Avg confidence of correct: {correct['confidence'].mean():.3f}")
    print(f"Avg strength of errors: {errors['strength'].mean():.3f}")
    print(f"Avg strength of correct: {correct['strength'].mean():.3f}")
    
    # Show hardest errors (highest confidence but wrong)
    errors_sorted = errors.sort_values('confidence', ascending=False)
    print(f"\n--- Top {min(top_n, len(errors))} High-Confidence Errors ---")
    for i, (idx, row) in enumerate(errors_sorted.head(top_n).iterrows(), 1):
        print(f"\n{i}. Gold: {row['gold_label']} → Pred: {row['pred_label']}")
        print(f"   Confidence: {row['confidence']:.3f}, Strength: {row['strength']:.3f}")
        print(f"   Scores: pos={row['pos_score_raw']:.3f}, neg={row['neg_score_raw']:.3f}")
        print(f"   Sentence: {row['sentence'][:150]}...")
    
    # Show lowest confidence errors (uncertain)
    errors_uncertain = errors.sort_values('confidence')
    print(f"\n--- Top {min(top_n, len(errors))} Low-Confidence Errors (Uncertain) ---")
    for i, (idx, row) in enumerate(errors_uncertain.head(top_n).iterrows(), 1):
        print(f"\n{i}. Gold: {row['gold_label']} → Pred: {row['pred_label']}")
        print(f"   Confidence: {row['confidence']:.3f}, Strength: {row['strength']:.3f}")
        print(f"   Scores: pos={row['pos_score_raw']:.3f}, neg={row['neg_score_raw']:.3f}")
        print(f"   Sentence: {row['sentence'][:150]}...")

# Run error analysis on best mode
best_eval_df = eval_results['exclude_gold_neutral']['df_eval']
analyze_errors(best_eval_df, top_n=3)

## 8. Complete Pipeline: Social Graph Generation

Now we'll create the complete pipeline from text to social graph.

In [ ]:
def create_social_graph(relations_df, output_json=None, output_png=None, title="Social Graph with Emotional Attribution"):
    """
    Create weighted, typed social graph from relations.
    
    Args:
        relations_df: DataFrame with columns [subject, object, pred_label, strength]
        output_json: Path to save JSON output
        output_png: Path to save PNG visualization
        title: Graph title
    
    Returns:
        (links, graph) tuple
    """
    # Create links in desired format
    links = []
    for _, row in relations_df.iterrows():
        links.append({
            "source": str(row['subject']),
            "target": str(row['object']),
            "linkType": str(row['pred_label']),
            "score": float(row['strength'])
        })
    
    # Save JSON
    if output_json:
        with open(output_json, 'w') as f:
            json.dump(links, f, indent=2)
        print(f"✓ Saved JSON: {output_json}")
    
    # Create NetworkX graph
    G = nx.DiGraph()
    for link in links:
        G.add_edge(
            link['source'], 
            link['target'],
            weight=link['score'],
            sentiment=link['linkType']
        )
    
    # Visualize
    plt.figure(figsize=(14, 10))
    
    # Layout
    if len(G.nodes()) <= 10:
        pos = nx.spring_layout(G, seed=42, k=3, iterations=50)
    else:
        pos = nx.kamada_kawai_layout(G)
    
    # Draw nodes
    nx.draw_networkx_nodes(
        G, pos, 
        node_size=3000, 
        node_color='lightblue',
        alpha=0.9, 
        edgecolors='darkblue', 
        linewidths=2.5
    )
    
    # Draw node labels
    nx.draw_networkx_labels(
        G, pos, 
        font_size=11, 
        font_weight='bold',
        font_color='black'
    )
    
    # Draw edges with colors and weights
    for u, v, data in G.edges(data=True):
        color = '#2ecc71' if data['sentiment'] == 'positive' else '#e74c3c'  # Green / Red
        width = 1 + 4 * data['weight']  # Scale by strength
        alpha = 0.3 + 0.6 * data['weight']
        
        nx.draw_networkx_edges(
            G, pos, 
            [(u, v)],
            edge_color=color,
            width=width,
            alpha=alpha,
            arrowsize=25,
            arrowstyle='->',
            connectionstyle='arc3,rad=0.1'
        )
    
    # Draw edge labels
    edge_labels = {
        (u, v): f"{data['sentiment']}\n({data['weight']:.2f})"
        for u, v, data in G.edges(data=True)
    }
    nx.draw_networkx_edge_labels(
        G, pos, 
        edge_labels=edge_labels,
        font_size=9,
        font_weight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='gray')
    )
    
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.axis('off')
    plt.tight_layout()
    
    # Save PNG
    if output_png:
        plt.savefig(output_png, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"✓ Saved PNG: {output_png}")
    
    plt.show()
    
    return links, G

# Example: Create graph from example text
example_text = "Bob likes Mary, Mary hates Jane, Jane adores and loves Bob very much."

# Parse relations (in practice, this comes from your DialogRE + DeBERTa pipeline)
example_relations = pd.DataFrame([
    {'subject': 'Bob', 'object': 'Mary', 'sentence': 'Bob likes Mary'},
    {'subject': 'Mary', 'object': 'Jane', 'sentence': 'Mary hates Jane'},
    {'subject': 'Jane', 'object': 'Bob', 'sentence': 'Jane adores and loves Bob very much'}
])

# Predict sentiments
for idx, row in example_relations.iterrows():
    pred = predict_sentiment(row['sentence'])
    example_relations.at[idx, 'pred_label'] = pred['pred_label']
    example_relations.at[idx, 'strength'] = pred['strength']

print("\nExample Relations with Predictions:")
print(example_relations[['subject', 'object', 'pred_label', 'strength']])
print()

# Generate graph
links, graph = create_social_graph(
    example_relations,
    output_json='/home/claude/example_social_graph.json',
    output_png='/home/claude/example_social_graph.png',
    title="Example: Social Graph with Emotional Attribution (Aigents)"
)

## 9. Full Pipeline on Test Data

In [ ]:
# Create graph from a subset of test data (positive/negative only, excluding neutrals)
test_subset = df_results[
    (df_results['gold_label'].isin(['positive', 'negative'])) & 
    (~df_results['is_likely_neutral'])
].head(10).copy()

print(f"Creating social graph from {len(test_subset)} test relations...\n")

links, graph = create_social_graph(
    test_subset,
    output_json='/home/claude/test_social_graph.json',
    output_png='/home/claude/test_social_graph.png',
    title="DialogRE Test Data: Social Graph with Emotional Attribution"
)

print(f"\nGraph statistics:")
print(f"  Nodes: {graph.number_of_nodes()}")
print(f"  Edges: {graph.number_of_edges()}")
print(f"  Positive edges: {sum(1 for _, _, d in graph.edges(data=True) if d['sentiment']=='positive')}")
print(f"  Negative edges: {sum(1 for _, _, d in graph.edges(data=True) if d['sentiment']=='negative')}")

## 10. Save Results

In [ ]:
# Save detailed results
output_path = '/home/claude/aigents_evaluation_results.csv'
df_results.to_csv(output_path, index=False)
print(f"✓ Saved detailed results: {output_path}")

# Save evaluation summary
summary_path = '/home/claude/aigents_evaluation_summary.json'
summary = {
    'total_samples': len(df_results),
    'gold_distribution': df_results['gold_label'].value_counts().to_dict(),
    'pred_distribution': df_results['pred_label'].value_counts().to_dict(),
    'detected_neutrals': int(df_results['is_likely_neutral'].sum()),
    'evaluations': {
        mode: {
            'n_samples': results['n_samples'],
            'accuracy': float(results['accuracy']),
            'f1_macro': float(results['f1_macro']),
            'f1_weighted': float(results['f1_weighted']),
            'precision_macro': float(results['precision_macro']),
            'recall_macro': float(results['recall_macro'])
        }
        for mode, results in eval_results.items()
    }
}

with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✓ Saved evaluation summary: {summary_path}")

print("\n" + "=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)
print("\n📊 Outputs generated:")
print("  1. Detailed predictions CSV")
print("  2. Evaluation summary JSON")
print("  3. Example social graph (JSON + PNG)")
print("  4. Test social graph (JSON + PNG)")
print("\n🎯 Key Results:")
print(f"  Best F1 (exclude neutrals): {eval_results['exclude_gold_neutral']['f1_macro']:.3f}")
print(f"  Best Accuracy: {eval_results['exclude_gold_neutral']['accuracy']:.3f}")
print("\n💡 Next Steps for AAAI Paper:")
print("  1. Compare with RoBERTa-GoEmotions, DeBERTa-v3, Llama-3")
print("  2. Analyze interpretability advantages of n-gram based approach")
print("  3. Evaluate on full DialogRE test set")
print("  4. Ablation study on neutral handling strategies")